In [1]:
#importing packages and vars

from pyspark.sql.session import SparkSession
import pyspark.sql.functions as f

import os
from dotenv import load_dotenv

#load environment variables
env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [2]:
#creating session

spark = (( 
          SparkSession
            .builder
            .appName('dim_delivery')
            .config('spark.jars', f'{sqlserverjar}')
            .config('spark.driver.extraClassPath', f'{sqlserverjar}')
            .config('spark.executor.extraClassPath', f'{sqlserverjar}')
            .getOrCreate()
        ))

print('Session created')

your 131072x1 screen size is bogus. expect trouble
24/04/12 16:20:26 WARN Utils: Your hostname, DOUGPC resolves to a loopback address: 127.0.1.1; using 172.25.197.48 instead (on interface eth0)
24/04/12 16:20:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
24/04/12 16:20:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/12 16:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/04/12 16:20:29 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Session created


In [7]:
tmp_deliveryframe = (( spark
                        .read
                        .format('jdbc')
                        .option('url', f'{mssqlsource}')
                        .option('driver', f'{mssqldriver}')
                        .option('user', f'{mssqluser}')
                        .option('password', f'{mssqlpwd}')
                        .option('query', 
                                
                                """ 
                                    select
                                    
                                        deliverymethodid,
                                        deliverymethodname
                                        
                                    from application.deliverymethods
                                
                                """)
                        
                    )).load()


In [8]:
deliveryframe = (( 
                    tmp_deliveryframe
                        .withColumnRenamed('deliverymethodid', 'deliveryid')
                        .withColumnRenamed('deliverymethodname', 'deliverytype')
                ))


In [10]:
(( deliveryframe
    .write
        .format('jdbc')
        .option('url', mssql_dwdest)
        .option('driver', mssqldriver)
        .option('user', mssqluser)
        .option('password', mssqlpwd)
        .option('dbtable', 'dimension.freight')
    .save(mode= 'overwrite')
))